In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%config InlineBackend.figure_format = 'retina'

import random

from nsppk import NSPPK

from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator


In [ ]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 250
min_num_nodes = 14
max_num_nodes = 16

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


In [ ]:
df = add(
    compose(name("cyc"), cycle()),
    compose(name("tree"), tree()),
)
decomposition_function = compose(intersection_edges(), df)
nbits = 14

neighbor_vectorizer = NSPPK(
    radius=1,
    distance=4,
    connector=1,
    nbits=14,
    parallel=True,
)

generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode="operator_hash", #label_mode: str = "operator_hash" (default) or "histogram" or "histogram_values" for AbstractGraph node labeling.
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=neighbor_vectorizer,
    n_jobs=1,
    debug=False,
    debug_level=1,
)


In [ ]:
%%time
generator.store(graphs, neighbor_vectorizer=neighbor_vectorizer)
print(f"stored_graphs = {len(generator.stored_graphs_)}")


## Local Conditional Generation

Sample one stored ZINC molecule at random, retrieve its nearest stored neighbors,
fit `ConditionalAutoregressiveGenerator` on that local set, and generate from
the sampled molecule's interpretation graph.

In [ ]:
%%time
n_neighbors = 30
n_samples = 7
n_instances_per_sample = 1

samples = generator.sample(
    n_samples=n_samples,
    n_instances_per_sample=n_instances_per_sample,
    n_neighbors=n_neighbors,
    random_state=None,
)

print(f"requested_seed_samples = {n_samples}")
print(f"instances_per_seed = {n_instances_per_sample}")
print(f"sampled_indices = {generator.last_sampled_indices_}")
print(f"last_neighbor_indices = {generator.last_neighbor_indices_}")
print(f"last_training_graphs = {len(generator.last_generation_training_graphs_)}")
print(f"generated = {len(samples)}")


In [ ]:
source_graphs = [generator.stored_graphs_[i] for i in generator.last_sampled_indices_]

print("Source molecules:")
_ = display_graphs(source_graphs, n_graphs_per_line=max(1, len(source_graphs)))

print("Generated molecules:")
_ = display_graphs(samples, n_graphs_per_line=max(1, n_samples * n_instances_per_sample))

print("Last local fitting set:")
_ = display_graphs(generator.last_generation_training_graphs_, n_graphs_per_line=8)


---